# Phase 32 -- three items closed off the standing backlog

Three things flagged as open since Phase 25/26/28 (also confirmed as still open in the
prior conversation turn):

1. **Did Phase 25a's `compute_gender_conditions` completeness-guard fix actually recover
   q15/q27's judge-score regression?** Answered here at zero cost, reusing data already
   on disk -- no new API calls needed for this part.
2. **Multi-turn conversation quality** (`eval/eval_multiturn_v1.jsonl`, 8 conversations) --
   built in an earlier phase, never executed.
3. **`eval_calc_v4` c09-c14** (overtime + leave-accrual calculators, built in Phase 26) --
   never executed.

Both (2) and (3) now also double as the first real test of the Phase 31 production
switch (`claude-haiku-4.5` + compact prompt) on conversation memory and the newer
calculators specifically, neither of which the Phase 29-31 re-review touched.

**Scoring approach, consistent with Phase 25's precedent:** no LLM judge for the
calculator questions (a structural check against `res["calculation"]` is more precise
and free) and no LLM judge for the multi-turn set either (8 conversations is few enough,
and each has a natural-language `expected_behavior` description that a person can read
directly against the transcript more reliably than trying to encode it as a judge
prompt). Given today's session already spent a fair amount on judge-calibration
problems, this keeps this round at generation-only cost.


In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
sys.path.insert(0, str(PROJECT))

from dotenv import load_dotenv
load_dotenv(PROJECT / ".env")

from src.retrieval import Retriever
from src.generation import get_production_generator, PRODUCTION_MODEL_ID

EVAL_DIR = PROJECT / "eval"
print("production model:", PRODUCTION_MODEL_ID)


production model: anthropic/claude-haiku-4.5


## 1. q15/q27 -- did Phase 25a's fix recover the regression? (zero new API cost)

Compares Phase 25's original judge scores (`eval/phase25_v3_judge.csv`, pre-Phase-25a-fix)
against the freshest available judge scores for the same two questions under the same
model/prompt (`eval/rereview_judge.csv`, Sonnet-5-judged `gpt-4o-mini` with the
*original* prompt, collected in Phase 31's re-review) -- both already on disk.

This does **not** reflect today's new production config (Haiku + compact prompt) --
it isolates the retrieval/calc changes since Phase 25 as the variable, holding model and
prompt fixed, which is what actually answers the question asked.


In [2]:
old = pd.read_csv(EVAL_DIR / "phase25_v3_judge.csv", encoding="utf-8-sig")
new = pd.read_csv(EVAL_DIR / "rereview_judge.csv", encoding="utf-8-sig")
new_prod = new[new.arm == "gpt-4o-mini (production)"]

for qid in ["q15", "q27"]:
    o = old[old.question_id == qid].iloc[0]
    n = new_prod[new_prod.question_id == qid].iloc[0]
    print(f"=== {qid} ===")
    print(f"  Phase 25  (pre-25a fix):  faithfulness {o.faithfulness_score}  relevancy {o.relevancy_score}")
    print(f"    faith reason: {o.faithfulness_reason}")
    print(f"    rel reason:   {o.relevancy_reason}")
    print(f"  Today     (post-25a fix, current retrieval, same model/prompt):  "
          f"faithfulness {n.faithfulness_score}  relevancy {n.relevancy_score}")
    print(f"    faith reason: {n.faithfulness_reason}")
    print(f"    rel reason:   {n.relevancy_reason}")
    print()


=== q15 ===
  Phase 25  (pre-25a fix):  faithfulness 4  relevancy 5
    faith reason: Most factual claims (contribution/age thresholds, hazardous profession rules, 2.5%/2% calculation) are accurately sourced, but the answer mislabels the 300-contribution/age-45 rule as part of 64(b) instead of 64(a)(2) and conflates the age-reduction table with the specific 64(g) reduction schedule, causing minor inaccuracies.
    rel reason:   The answer directly confirms early retirement is possible and explains eligibility conditions and pension calculation methods, fully addressing the question.
  Today     (post-25a fix, current retrieval, same model/prompt):  faithfulness 4  relevancy 4
    faith reason: The answer accurately summarizes Article 64's eligibility conditions and calculation method, though it omits the age-based reduction table and dependent increases while still being consistent with the source.
    rel reason:   It directly answers both parts of the question (early retirement eligi

**Result: q27 recovered cleanly (3/3 -> 5/5, both metrics). q15 did not -- it's a
different, milder problem now, not the same one, but the score itself (5 relevancy in
Phase 25) didn't come back; it's 4/4 today.** Phase 25's q15 complaint was a specific
mislabelling (300-contribution rule attributed to the wrong sub-clause, table conflated
with the wrong reduction schedule) -- that exact defect is gone from today's reason.
What's there instead is an omission (age-based reduction table and dependent increases
left out), costing relevancy a point it didn't lose before. **Read as: Phase 25a's guard
fixed what it targeted (the completeness/relevance gate on `compute_gender_conditions`);
q15's remaining gap is a different, ordinary coverage omission, unrelated to that fix,**
and stays open as exactly that -- a completeness issue, not a regression.


## 2. Multi-turn conversation eval (`eval/eval_multiturn_v1.jsonl`, 8 conversations)

Replays each conversation's fixed `history` and asks `final_question`, following the
**exact** production flow in `src/api.py`'s `/ask` endpoint: `rewrite_for_retrieval` when
history exists, retrieve on the rewritten query, generate on the original question with
history. 3 trials per conversation (Phase 23a's lesson: a sampling model needs repeated
trials, not one, to trust a behavioural read). Runs under today's actual production
config (`claude-haiku-4.5` + compact prompt) -- this is also that config's first test on
multi-turn behaviour specifically.


In [3]:
MT_PATH = EVAL_DIR / "multiturn_v1_results.csv"
N_TRIALS = 3

mt = pd.read_json(EVAL_DIR / "eval_multiturn_v1.jsonl", lines=True)
retriever = Retriever()
gen = get_production_generator()

mt_rows = []
if MT_PATH.exists():
    mt_rows = pd.read_csv(MT_PATH, encoding="utf-8-sig").to_dict("records")
done = {}
for r in mt_rows:
    done[r["conversation_id"]] = done.get(r["conversation_id"], 0) + 1

for row in mt.itertuples(index=False):
    have = done.get(row.conversation_id, 0)
    for trial in range(have, N_TRIALS):
        history = list(row.history)
        retrieval_query = gen.rewrite_for_retrieval(row.final_question, history) if history else row.final_question
        chunks = retriever.retrieve(retrieval_query, top_k=5)
        res = gen.generate(row.final_question, chunks, history=history)
        gold = set(row.answer_article_numbers)
        retrieved = {c["article_number"] for c in chunks}
        mt_rows.append({
            "conversation_id": row.conversation_id, "trial": trial, "category": row.category,
            "final_question": row.final_question, "retrieval_query": retrieval_query,
            "gold_articles": sorted(gold), "retrieved_articles": sorted(retrieved),
            "gold_present": gold.issubset(retrieved), "answer": res["answer"],
        })
        pd.DataFrame(mt_rows).to_csv(MT_PATH, index=False, encoding="utf-8-sig")
        print(f"  {row.conversation_id} / trial {trial}: gold_present={gold.issubset(retrieved)}")

mt_df = pd.DataFrame(mt_rows)
print(f"\n{len(mt_df)} rows (expect {len(mt) * N_TRIALS})")


  mt01 / trial 0: gold_present=True


  mt01 / trial 1: gold_present=True


  mt01 / trial 2: gold_present=True


  mt02 / trial 0: gold_present=True


  mt02 / trial 1: gold_present=True


...

  mt07 / trial 1: gold_present=True


  mt07 / trial 2: gold_present=True


  mt08 / trial 0: gold_present=True


  mt08 / trial 1: gold_present=True


  mt08 / trial 2: gold_present=True

24 rows (expect 24)


In [4]:
print("Retrieval: gold article(s) present after history-aware rewrite, by conversation:")
print(mt_df.groupby("conversation_id").gold_present.agg(["sum", "count"]))


Retrieval: gold article(s) present after history-aware rewrite, by conversation:
                 sum  count
conversation_id            
mt01               3      3
mt02               3      3
mt03               3      3
mt04               3      3
mt05               3      3
mt06               3      3
mt07               3      3
mt08               3      3


In [5]:
for cid in mt.conversation_id:
    sub = mt_df[mt_df.conversation_id == cid]
    expected = mt.loc[mt.conversation_id == cid, "expected_behavior"].iloc[0]
    print("=" * 80)
    print(cid, "--", mt.loc[mt.conversation_id == cid, "category"].iloc[0])
    print("expected:", expected)
    print()
    for _, r in sub.iterrows():
        print(f"--- trial {r.trial} (retrieval query: {r.retrieval_query!r}) ---")
        print(r.answer[:500])
        print()


mt01 -- multiturn_fragment_retrieval
expected: The bare reply carries no semantic signal about end-of-service pay on its own. rewrite_for_retrieval must fold the history into a standalone query (something like 'end-of-service pay if not subject to Social Security Law') so retrieval still lands on Article 32, not unrelated articles.

--- trial 0 (retrieval query: 'كم مكافأة نهاية الخدمة التي أستحقها إذا لم أكن خاضعاً لقانون الضمان الاجتماعي؟') ---
إذاً أنت تستحق مكافأة نهاية الخدمة بموجب **قانون العمل لسنة 1996، المادة (32)**.

لكن لحساب المبلغ بدقة، أحتاج منك معلومتين:

1. **كم عدد سنوات خدمتك الفعلية؟**
2. **ما آخر أجر شهري تقاضيته؟** (أو إذا كان الأجر كله أو بعضه على أساس عمولة أو قطعة، فما متوسط الأجر الشهري خلال آخر 12 شهراً قبل انتهاء خدمتك)

**القاعدة الأساسية:** مكافأة نهاية الخدمة = **أجر شهر واحد عن كل سنة خدمة كاملة**، مع استحقاق مكافأة نسبية عن كسور السنة.

معلومات إضافية:
- **الفواصل بين عمل وآخر** لا تزيد على 60 يوماً تُحسب 

--- trial 1 (retrieval query: 'كم مكافأة نهاية 

**Read every conversation above against its `expected_behavior` line before writing the
verdict below.** This is a manual read, not a scripted pass/fail -- 8 conversations x 3
trials is a small enough set to read directly, and per this project's own repeated
lesson (Phase 23a), a keyword-matching detector for something like "did it ask" or "did
it use the right figure" is easy to get subtly wrong and silently misscore a whole run.


## 3. `eval_calc_v4` c09-c14 -- overtime and leave-accrual calculators

Structural check against `res["calculation"]` plus a narration read, same as Phase 25/26.
Also the compact-prompt/Haiku config's first test of these two calculators.


In [6]:
CALC_PATH = EVAL_DIR / "calc_v4_c09_c14_results.csv"
CALC_QUESTIONS = ["c09", "c10", "c11", "c12", "c13", "c14"]

ec = pd.read_json(EVAL_DIR / "eval_calc_v4.jsonl", lines=True)
ec = ec[ec.question_id.isin(CALC_QUESTIONS)]

calc_rows = []
if CALC_PATH.exists():
    calc_rows = pd.read_csv(CALC_PATH, encoding="utf-8-sig").to_dict("records")
cdone = {}
for r in calc_rows:
    cdone[r["question_id"]] = cdone.get(r["question_id"], 0) + 1

for row in ec.itertuples(index=False):
    have = cdone.get(row.question_id, 0)
    chunks = retriever.retrieve(row.question_text, top_k=5)
    for trial in range(have, N_TRIALS):
        res = gen.generate(row.question_text, chunks)
        calc_rows.append({
            "question_id": row.question_id, "trial": trial, "question_text": row.question_text,
            "labeling_note": row.labeling_note, "calculation": res.get("calculation"), "answer": res["answer"],
        })
        pd.DataFrame(calc_rows).to_csv(CALC_PATH, index=False, encoding="utf-8-sig")
        print(f"  {row.question_id} / trial {trial}: calc fired = {bool(res.get('calculation'))}")

calc_df = pd.DataFrame(calc_rows)
print(f"\n{len(calc_df)} rows (expect {len(CALC_QUESTIONS) * N_TRIALS})")


  c09 / trial 0: calc fired = True


  c09 / trial 1: calc fired = True


  c09 / trial 2: calc fired = True


  c10 / trial 0: calc fired = True


  c10 / trial 1: calc fired = True


...

  c13 / trial 1: calc fired = True


  c13 / trial 2: calc fired = True


  c14 / trial 0: calc fired = True


  c14 / trial 1: calc fired = True


  c14 / trial 2: calc fired = True

18 rows (expect 18)


In [7]:
for qid in CALC_QUESTIONS:
    sub = calc_df[calc_df.question_id == qid]
    note = ec.loc[ec.question_id == qid, "labeling_note"].iloc[0]
    print("=" * 80)
    print(qid, "-- expected:", note)
    print()
    for _, r in sub.iterrows():
        print(f"--- trial {r.trial} -- calc: {r.calculation} ---")
        print(r.answer[:400])
        print()


c09 -- expected: Phase 26, live-verified over real HTTP. Explicit HOURLY wage stated -> expect the 125% ordinary-overtime rate applied: 3.75 JD/hour, 7.50 JD for the two hours.

--- trial 0 -- calc: {'overtime': {'law': 'labor96', 'article': '59', 'language': 'ar', 'ordinary_rate': 125.0, 'holiday_rate': 150.0, 'hourly_wage': 3.0, 'ordinary_amount': 3.75, 'holiday_amount': 4.5, 'requires_not_excluded_role': False}} ---
بناءً على قانون العمل لسنة 1996، المادة (59)، يجب أن تتقاضى عن العمل الإضافي أجراً لا يقل عن 125% من أجرك المعتاد.

**حسابك:**
- أجرك بالساعة: 3 دينار
- نسبة العمل الإضافي: 125%
- أجر الساعة الإضافية: 3 × 125% = **3.75 دينار**
- عن ساعتين إضافيتين: 3.75 × 2 = **7.5 دينار**

لكن هذا يتوقف على **متى عملت هاتان الساعتان الإضافيتان:**
- إذا كانت في أيام عملك العادية: تستحق **7.5 دينار**
- إذا كانت في 

--- trial 1 -- calc: {'overtime': {'law': 'labor96', 'article': '59', 'language': 'ar', 'ordinary_rate': 125.0, 'holiday_rate': 150.0, 'hourly_wage': 3.0, 'ordinary_amount': 3

## Verdict

### 1. q15/q27 -- answered above, zero cost. q27 recovered cleanly (3/3 -> 5/5). q15 did not -- but the reason it didn't is a different, milder, unrelated issue (an omission), not the original defect recurring. Phase 25a's fix did what it was built to do.

### 2. Multi-turn (8 conversations x 3 trials, 24 runs) -- retrieval 24/24, generation 7/8 clean, 1 real gap found

**Retrieval (the history-aware rewrite mechanism itself): 24/24 -- every trial, every conversation landed on the gold article(s) despite bare fragments, one-word replies, and facts stated one or two turns earlier.** This is the core Phase 23 mechanism and it held perfectly under the new model.

**Generation, read against each `expected_behavior` line:**
- **mt01** (fragment retrieval, end-of-service): 3/3 correctly grounded in Article 32 and correctly conditioned on "not subject to Social Security". Minor stylistic variation (2/3 ask for years/wage, 1/3 explains the formula without asking) -- not a failure, both are reasonable given the numbers genuinely aren't known.
- **mt02** (sex stated as a one-word reply, "Male") -- **the one real gap.** Only 1 of 3 trials correctly resolved to the male-specific figure (252 contributions, explicitly "Based on your situation as a male"). The other 2 trials listed every pathway in Article 64 (main, transitional, hazardous-profession, newly-covered) without ever narrowing to the stated fact -- not a *wrong* answer (neither stated the female 228 figure), but a **missed** one: the person told the model their sex in this exact turn, and 2 of 3 times the answer didn't use it. A completeness gap, not a silent-wrong-default -- but still short of what Phase 24e/25a's mechanism is supposed to guarantee.
- **mt03** (sex carried forward from 2 turns back): 3/3 -- correctly used the female figure, correctly addressed the person as "أنتِ", never re-asked.
- **mt04** (sex must NOT be read from the assistant's own earlier "if male/if female" recitation): 3/3 -- every trial treated sex as still unknown, presented both cases or asked, never silently picked one because the assistant itself had said both words earlier.
- **mt05** (no over-asking, institutional follow-up): 3/3 -- answered directly from Articles 57/58, no clarifying question.
- **mt06** (ask on current turn despite an earlier turn touching the same article): 3/3 -- correctly re-evaluated fresh (230 contributions sits between the female threshold, met, and the male threshold, not met), presented both cases rather than treating the topic as already resolved.
- **mt07** (salary stated one turn earlier): 3/3 -- the contributions calculator fired correctly off a fact from history, all three trials computed the exact right 142.5 JD breakdown.
- **mt08** (Hajj leave, same article as already cited): 3/3 -- answered directly, correctly recognised Hajj as one of Article 27(a)(3)'s protected periods, even correctly volunteered the 27(b) exception.

**7 of 8 mechanisms held perfectly across all 3 trials. mt02 is a real, specific, reproducible gap** -- worth a follow-up (likely `extract_sex` not reliably parsing a single-word "Male" reply the way it parses sentence-form statements, or the resulting block not being narrated consistently) rather than something to re-run and hope goes away.

### 3. `eval_calc_v4` c09-c14 -- 6/6 questions, 18/18 trials correct, both guardrails held

- **c09** (explicit hourly wage): 3/3 -- 3.75 JD/hour, 7.50 JD for two hours, exactly as expected.
- **c10** (no wage stated): 3/3 -- states the 125% rate only, no invented JD figure.
- **c11** (holiday/day-off rate specifically): 3/3 -- correctly gives 150%, never the wrong 125% ordinary rate.
- **c12** (the monthly-salary guardrail): **3/3 -- the calculator declined to invent an hourly figure from the 900 JD monthly salary in every trial**, and the model's own narration explicitly flagged the gap ("the documents don't specify how to convert a monthly salary to a daily/hourly one") rather than quietly making up a number. This is the exact failure this question exists to catch, and it didn't happen once.
- **c13** (leave, below the 5-year threshold): 3/3 -- exact 42 days (14 x 3), correctly explained.
- **c14** (the blended-total guardrail, Phase 26's live-caught bug verbatim): **3/3 -- no trial invented a blended total.** Every one stated both rates (14/21), the 5-year threshold, and said plainly that the documents don't specify exactly when the threshold was crossed during the 8 years -- the precise discipline Phase 26 built this guard to enforce, and it held under a different model with a different (compact) prompt.

### Overall

Two of the three open items are now fully closed with clean results (q27's recovery confirmed, calc_v4's two hardest guardrail questions held perfectly under the new production config). The multi-turn set surfaced one real, specific, reproducible gap (mt02) worth a follow-up fix -- not a reason to doubt the mechanism overall, since the other 7 of 8 behaviors it protects held across every trial. q15 stays open as an ordinary, unrelated completeness gap, not a regression.
